In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import shap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tensorflow.keras.preprocessing import image

# Load trained model
model = tf.keras.models.load_model("aocdml.keras")
last_conv_layer_name = "conv2d_2"

In [ ]:
# Functional API for Grad-CAM
def api_gradcam_model(model, last_conv_layer_name):
    inputs = tf.keras.Input(shape=(224, 224, 1))
    x = inputs
    conv_output = None

    for layer in model.layers:
        x = layer(x)
        if layer.name == last_conv_layer_name:
            conv_output = x

    grad_model = tf.keras.Model(inputs=inputs, outputs=[conv_output, x])
    return grad_model

In [ ]:
# Grad-CAM function
def make_gradcam_heatmap(img_array, grad_model, pred_index):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, pred_index]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.math.reduce_max(heatmap) + 1e-8  
    return heatmap.numpy()

In [ ]:
# SHAP function
def make_shap_explanation(img_array, model):
    background_data = np.load("background_data.npy")
    if background_data.ndim == 3:
        background_data = np.expand_dims(background_data, axis=-1)
    if img_array.ndim == 3:
        img_array = np.expand_dims(img_array, axis=0)
    explainer = shap.GradientExplainer(model, background_data)
    shap_values = explainer.shap_values(img_array)
    return shap_values

In [ ]:
# def crop_and_overwrite(image_path):
#     img = cv2.imread(image_path)
#     if img is None:
#         print(f"Error loading: {image_path}")
#         return False
        
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
#     contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

#     if not contours:
#         print(f"No contours found in: {image_path}")
#         return False
#     largest_contour = max(contours, key=cv2.contourArea)
#     mask = np.zeros_like(gray)
#     cv2.drawContours(mask, [largest_contour], -1, 255, thickness=cv2.FILLED)
#     masked_img = cv2.bitwise_and(img, img, mask=mask)
#     x, y, w, h = cv2.boundingRect(largest_contour)
#     final_cropped_img = masked_img[y:y+h, x:x+w]

#     cv2.imwrite(image_path, final_cropped_img)
#     return True

In [ ]:
# Load image
grad_model = api_gradcam_model(model, last_conv_layer_name)
img_path = "check_XAI/img.jpg"
# success = crop_and_overwrite(img_path)
# if success:
    # print(f"Successfully cropped and overwritten: {img_path}")
img = image.load_img(img_path, target_size=(224,224), color_mode="grayscale")
img_array = image.img_to_array(img)  
img_array = np.expand_dims(img_array, axis=0).astype(np.float32)
img_array /= 255.0  

# Make prediction
prediction = model.predict(img_array)
predicted_class = np.argmax(prediction[0])

# Grad-CAM heatmap
heatmap = make_gradcam_heatmap(img_array, grad_model, predicted_class)
if (np.max(heatmap) == 0.0):
    grad_model = api_gradcam_model(model, "conv2d_1")
    img_array = img_array + np.random.normal(0, 0.01, img_array.shape)
    heatmap = make_gradcam_heatmap(img_array, grad_model, predicted_class)
heatmap = cv2.resize(heatmap, (img.size[0], img.size[1]))
heatmap = np.uint8(255 * heatmap)
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

img_display = np.uint8(255.0 * img_array[0].squeeze())
img_display = cv2.cvtColor(img_display, cv2.COLOR_GRAY2BGR)
superimposed_img = cv2.addWeighted(img_display, 0.6, heatmap, 0.4, 0)
# superimposed_img = cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB)

# SHAP values
shap_values = make_shap_explanation(img_array, model)
if isinstance(shap_values, list):
    shap_map = shap_values[predicted_class][0]
else:
    shap_map = shap_values[0, ..., predicted_class]

shap_map = shap_map.squeeze()
max_val = np.max(np.abs(shap_map))
masked_shap = np.copy(shap_map)
threshold = max_val * 0.10 
masked_shap[np.abs(shap_map) < threshold] = np.nan

# Return prediction
percentages = (prediction[0] * 100).round(2)
if predicted_class == 0:
    category = "Benign Ovarian Tumor"
    probability = percentages[0]
elif predicted_class == 1:
    category = "Normal Ovary"
    probability = percentages[1]
elif predicted_class == 2:
    category = "Ovarian Cancer"
    probability = percentages[2]
elif predicted_class == 3:
    category = "Uterine Fibroid"
    probability = percentages[3]

print("Predicted class:", category)
print("Probability:", probability)

# Grad-CAM overlay
plt.imshow(superimposed_img)
plt.axis("off")
plt.title("Grad-CAM  \nLighter regions indicate stronger influence on the prediction")
plt.show()

# SHAP explanation
plt.figure(figsize=(5, 5))
plt.imshow(img_array[0].squeeze(), cmap='gray')
plt.imshow(masked_shap, cmap='seismic', vmin=-max_val, vmax=max_val, alpha=0.7)
plt.axis('off')
plt.title(f"SHAP Overlay: \n(Red = Increases Risk, Blue = Decreases Risk)")
plt.colorbar(label="SHAP Value impact", fraction=0.046, pad=0.04)
plt.show()



In [ ]:
# Combined visualization

plt.figure(figsize=(15, 5))
    
plt.subplot(1, 3, 1)
plt.imshow(img_array[0].squeeze(), cmap='gray')
plt.title("Original Image", fontsize=10)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(img_array[0].squeeze(), cmap='gray')
plt.imshow(masked_shap, cmap='seismic', vmin=-max_val, vmax=max_val, alpha=0.7)
plt.axis('off')
plt.title(f"SHAP Overlay: \n(Red = Increases Risk, Blue = Decreases Risk)", fontsize=10)
plt.colorbar(label="SHAP Value impact", fraction=0.046, pad=0.04)

plt.subplot(1, 3, 3)
plt.imshow(superimposed_img)
plt.axis("off")
plt.title("Grad-CAM  \nLighter regions indicate stronger influence on the prediction", fontsize=10)

plt.tight_layout()
plt.suptitle(f"Model Prediction: {category} ({probability:.2f}%)\n", fontsize=12)
plt.show()